# Haiti vs Scotland: Match Visualisations in Python

This notebook recreates the two proposed article graphics:

1. A cumulative expected-goals (xG) timeline.
2. A numbered map of Scotland's decisive goal sequence.

The outputs follow the Sports Data Campus palette and are exported at **1200 × 800 pixels** as compressed JPG files under 500 KB.

Required packages: `pandas`, `numpy`, `matplotlib`, and `Pillow`.

If needed, install them in a separate notebook cell with:

```python
%pip install pandas numpy matplotlib pillow
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Arc, Circle, FancyArrowPatch, FancyBboxPatch, Rectangle
from PIL import Image

warnings.filterwarnings('ignore', category=UserWarning)

# Resolve paths whether the notebook is launched from the project root or /analysis.
CWD = Path.cwd().resolve()
if (CWD / 'visual_data').exists():
    ANALYSIS_DIR = CWD
elif (CWD / 'analysis' / 'visual_data').exists():
    ANALYSIS_DIR = CWD / 'analysis'
else:
    raise FileNotFoundError('Could not locate analysis/visual_data. Open the notebook inside the project folder.')

DATA_DIR = ANALYSIS_DIR / 'visual_data'
OUTPUT_DIR = ANALYSIS_DIR / 'visuals_python'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sports Data Campus approved palette.
COLORS = {
    'scotland': '#1F77B4',
    'haiti': '#D62728',
    'goal': '#FF7F0E',
    'carry': '#17BECF',
    'saved_shot': '#9467BD',
    'green': '#2CA02C',
    'ink': '#14213D',
    'muted': '#5B6577',
    'grid': '#D9E1E8',
    'background': '#F7F8F5',
    'pitch': '#F1F4ED',
    'white': '#FFFFFF',
}

# Bundle the approved typography with the project so the notebook is portable.
FONT_DIR = ANALYSIS_DIR / 'fonts'
TITLE_FONT_PATH = FONT_DIR / 'BarlowCondensed-Bold.ttf'
BODY_FONT_PATH = FONT_DIR / 'OpenSans-Variable.ttf'
for font_path in [TITLE_FONT_PATH, BODY_FONT_PATH]:
    if not font_path.exists():
        raise FileNotFoundError(f'Missing required font: {font_path}')
    font_manager.fontManager.addfont(str(font_path))

TITLE_FONT = font_manager.FontProperties(fname=str(TITLE_FONT_PATH)).get_name()
BODY_FONT = font_manager.FontProperties(fname=str(BODY_FONT_PATH)).get_name()

mpl.rcParams.update({
    'font.family': BODY_FONT,
    'text.color': COLORS['ink'],
    'axes.labelcolor': COLORS['muted'],
    'xtick.color': COLORS['muted'],
    'ytick.color': COLORS['muted'],
    'figure.facecolor': COLORS['background'],
    'axes.facecolor': COLORS['background'],
})

print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)

## Load the prepared visual datasets

`viz_1_xg_timeline.csv` contains every shot and its cumulative team xG.  
`viz_2_goal_sequence.csv` contains the passes, carries, recovery, and shots involved in Scotland's goal.

In [ ]:
shots = pd.read_csv(DATA_DIR / 'viz_1_xg_timeline.csv')
goal_sequence = pd.read_csv(DATA_DIR / 'viz_2_goal_sequence.csv')

numeric_shot_columns = ['minute', 'second', 'match_minute', 'xg', 'cumulative_xg', 'start_x', 'start_y', 'end_x', 'end_y']
shots[numeric_shot_columns] = shots[numeric_shot_columns].apply(pd.to_numeric, errors='coerce')

numeric_sequence_columns = ['sequence_order', 'minute', 'second', 'start_x', 'start_y', 'end_x', 'end_y', 'pass_length', 'xg']
goal_sequence[numeric_sequence_columns] = goal_sequence[numeric_sequence_columns].apply(pd.to_numeric, errors='coerce')

display(shots[['minute', 'second', 'team', 'player', 'xg', 'cumulative_xg', 'outcome']])
display(goal_sequence[['sequence_order', 'event_type', 'player', 'recipient', 'start_x', 'start_y', 'end_x', 'end_y', 'xg']])

## Shared export helper

The figure is first saved as a lossless PNG and then compressed to JPG using Pillow. This keeps the dimensions fixed while controlling the final file size.

In [ ]:
def export_jpg(fig, filename, quality=88):
    """Export a 12 × 8 inch figure at 100 dpi as a compressed 1200 × 800 JPG."""
    png_path = OUTPUT_DIR / filename.replace('.jpg', '_temporary.png')
    jpg_path = OUTPUT_DIR / filename

    fig.savefig(
        png_path,
        dpi=100,
        facecolor=fig.get_facecolor(),
        edgecolor='none',
        bbox_inches=None,
        pad_inches=0,
    )
    with Image.open(png_path) as image:
        image.convert('RGB').save(
            jpg_path,
            format='JPEG',
            quality=quality,
            optimize=True,
            progressive=True,
            subsampling=0,
        )
    png_path.unlink(missing_ok=True)

    with Image.open(jpg_path) as check:
        size_kb = jpg_path.stat().st_size / 1024
        print(f'{jpg_path.name}: {check.width} × {check.height}px, {size_kb:.1f} KB')
    return jpg_path

## Visual 1: Cumulative xG and match momentum

The step lines rise after every shot. The annotations focus on Scotland's decisive first-half sequence and Haiti's two largest opportunities.

In [ ]:
fig = plt.figure(figsize=(12, 8), dpi=100, facecolor=COLORS['background'])
ax = fig.add_axes([0.075, 0.19, 0.855, 0.57])

team_settings = {
    'Scotland': COLORS['scotland'],
    'Haiti': COLORS['haiti'],
}

for team, color in team_settings.items():
    team_shots = shots.loc[shots['team'].eq(team)].sort_values('match_minute')
    x = np.r_[0, team_shots['match_minute'].to_numpy(), 100]
    y = np.r_[0, team_shots['cumulative_xg'].to_numpy(), team_shots['cumulative_xg'].iloc[-1]]
    ax.step(x, y, where='post', color=color, linewidth=4, zorder=3)
    ax.scatter(
        team_shots['match_minute'],
        team_shots['cumulative_xg'],
        s=36,
        color=color,
        edgecolor=COLORS['white'],
        linewidth=1.3,
        zorder=4,
    )

# Highlight the closing phase.
ax.axvspan(75, 100, color=COLORS['haiti'], alpha=0.055, zorder=0)

# Goal marker and line.
goal = shots.loc[shots['is_goal'].astype(str).str.lower().eq('true')].iloc[0]
ax.axvline(goal['match_minute'], color=COLORS['goal'], linestyle=(0, (4, 4)), linewidth=1.8, zorder=1)
ax.scatter(goal['match_minute'], goal['cumulative_xg'], s=190, color=COLORS['goal'], edgecolor=COLORS['white'], linewidth=3, zorder=6)
ax.text(goal['match_minute'] + 1.2, 1.31, "GOAL 28'", color=COLORS['goal'], fontsize=11, fontweight='bold')

# Key shot lookups.
pierrot_blocked = shots.loc[(shots['player'].str.contains('Pierrot')) & shots['outcome'].eq('Blocked')].iloc[0]
pierrot_late = shots.loc[(shots['player'].str.contains('Pierrot')) & shots['match_minute'].gt(90)].iloc[0]

# Put the three incident cards in a dedicated band above the data lines.
box = dict(boxstyle='round,pad=0.65', facecolor=COLORS['white'], edgecolor=COLORS['grid'], linewidth=1)
ax.annotate(
    "SCOTLAND'S DECISIVE BURST\nAdams: 0.40 xG saved | McGinn: 0.07 xG goal",
    xy=(goal['match_minute'], goal['cumulative_xg']),
    xytext=(17, 1.55),
    fontsize=9.2,
    fontweight='bold',
    ha='center',
    va='top',
    linespacing=1.35,
    bbox=box,
    arrowprops=dict(arrowstyle='-', color=COLORS['scotland'], linestyle=(0, (2, 3)), linewidth=1.1, connectionstyle='arc3,rad=-0.08'),
    zorder=7,
)
ax.annotate(
    "PIERROT, 34'\n0.42 xG effort blocked",
    xy=(pierrot_blocked['match_minute'], pierrot_blocked['cumulative_xg']),
    xytext=(50, 1.55),
    fontsize=9.2,
    fontweight='bold',
    ha='center',
    va='top',
    linespacing=1.35,
    bbox=box,
    arrowprops=dict(arrowstyle='-', color=COLORS['haiti'], linestyle=(0, (2, 3)), linewidth=1.1, connectionstyle='arc3,rad=0.08'),
    zorder=7,
)
ax.annotate(
    "PIERROT, 90+3'\n0.31 xG attempt saved",
    xy=(pierrot_late['match_minute'], pierrot_late['cumulative_xg']),
    xytext=(82, 1.55),
    fontsize=9.2,
    fontweight='bold',
    ha='center',
    va='top',
    linespacing=1.35,
    bbox=box,
    arrowprops=dict(arrowstyle='-', color=COLORS['haiti'], linestyle=(0, (2, 3)), linewidth=1.1, connectionstyle='arc3,rad=-0.06'),
    zorder=7,
)

ax.set_xlim(0, 100)
ax.set_ylim(0, 1.65)
ax.set_xticks([0, 15, 30, 45, 60, 75, 90, 100], ["0'", "15'", "30'", "45'", "60'", "75'", "90'", "100'"])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])
ax.grid(color=COLORS['grid'], linewidth=0.8, alpha=0.9)
ax.set_xlabel('MATCH MINUTE', fontsize=10, fontweight='bold', labelpad=12)
ax.set_ylabel('CUMULATIVE xG', fontsize=10, fontweight='bold', labelpad=12)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0, labelsize=9)

# Figure-level heading and legend.
fig.text(0.05, 0.92, 'SCOTLAND STRIKE EARLY, HAITI BUILD THE PRESSURE', fontsize=25, fontweight='bold', fontfamily=TITLE_FONT)
fig.text(0.05, 0.875, 'Cumulative event-level xG timeline | Haiti 0-1 Scotland', fontsize=12, color=COLORS['muted'])

scotland_total = shots.loc[shots['team'].eq('Scotland'), 'cumulative_xg'].iloc[-1]
haiti_total = shots.loc[shots['team'].eq('Haiti'), 'cumulative_xg'].iloc[-1]
fig.text(0.075, 0.81, f'━  Scotland {scotland_total:.2f} xG', color=COLORS['scotland'], fontsize=11, fontweight='bold')
fig.text(0.255, 0.81, f'━  Haiti {haiti_total:.2f} xG', color=COLORS['haiti'], fontsize=11, fontweight='bold')
fig.text(0.435, 0.81, '●  Goal', color=COLORS['goal'], fontsize=11)

fig.text(
    0.615,
    0.82,
    "FROM 75' ONWARD\nHaiti: 6 shots, 0.58 xG | Scotland: 0 shots",
    fontsize=9.5,
    fontweight='bold',
    color=COLORS['haiti'],
    linespacing=1.4,
    bbox=dict(boxstyle='round,pad=0.8', facecolor='#FDECEC', edgecolor='none'),
)

fig.text(0.05, 0.055, 'Data: event-level StatsBomb shot xG. Totals may differ slightly from the aggregate match feed.', fontsize=8.5, color=COLORS['muted'])
fig.text(0.95, 0.055, 'El Mundial de los Datos', fontsize=8.5, color=COLORS['muted'], fontweight='bold', ha='right')

xg_output = export_jpg(fig, 'Haiti_vs_Scotland_xG_Momentum_Timeline_Python.jpg')
plt.show()

## Visual 2: Scotland's 26-second goal sequence

The pitch is drawn directly with Matplotlib. Blue arrows represent completed passes, cyan dashed arrows show the key carries, purple marks Adams' saved shot, and orange marks McGinn's goal.

In [ ]:
def draw_pitch(ax):
    """Draw a 120 × 80 StatsBomb-style football pitch."""
    line_color = '#B7C2B4'
    ax.set_facecolor(COLORS['pitch'])
    ax.add_patch(Rectangle((0, 0), 120, 80, fill=False, edgecolor='#9BAA9A', linewidth=1.5))
    ax.plot([60, 60], [0, 80], color=line_color, linewidth=1.3)
    ax.add_patch(Circle((60, 40), 9.15, fill=False, edgecolor=line_color, linewidth=1.3))
    ax.add_patch(Circle((60, 40), 0.55, color=line_color))

    ax.add_patch(Rectangle((0, 18), 18, 44, fill=False, edgecolor=line_color, linewidth=1.3))
    ax.add_patch(Rectangle((102, 18), 18, 44, fill=False, edgecolor=line_color, linewidth=1.3))
    ax.add_patch(Rectangle((0, 30), 6, 20, fill=False, edgecolor=line_color, linewidth=1.3))
    ax.add_patch(Rectangle((114, 30), 6, 20, fill=False, edgecolor=line_color, linewidth=1.3))
    ax.add_patch(Circle((12, 40), 0.55, color=line_color))
    ax.add_patch(Circle((108, 40), 0.55, color=line_color))
    ax.add_patch(Arc((12, 40), 18.3, 18.3, theta1=310, theta2=50, color=line_color, linewidth=1.3))
    ax.add_patch(Arc((108, 40), 18.3, 18.3, theta1=130, theta2=230, color=line_color, linewidth=1.3))
    ax.add_patch(Rectangle((-1.5, 36), 1.5, 8, fill=False, edgecolor='#8B9790', linewidth=1.3))
    ax.add_patch(Rectangle((120, 36), 1.5, 8, fill=False, edgecolor='#8B9790', linewidth=1.3))

    ax.set_xlim(-2, 122)
    ax.set_ylim(82, -2)
    ax.set_aspect('equal')
    ax.axis('off')


def plot_arrow(ax, row, color, linewidth=2.4, linestyle='-', zorder=3):
    arrow = FancyArrowPatch(
        (row.start_x, row.start_y),
        (row.end_x, row.end_y),
        arrowstyle='-|>',
        mutation_scale=13,
        linewidth=linewidth,
        linestyle=linestyle,
        color=color,
        shrinkA=1,
        shrinkB=1,
        zorder=zorder,
    )
    ax.add_patch(arrow)


fig = plt.figure(figsize=(12, 8), dpi=100, facecolor=COLORS['background'])
pitch_ax = fig.add_axes([0.035, 0.10, 0.675, 0.66])
side_ax = fig.add_axes([0.73, 0.10, 0.235, 0.66])
draw_pitch(pitch_ax)
side_ax.axis('off')

passes = goal_sequence.loc[goal_sequence['event_type'].eq('Pass')].reset_index(drop=True)
carries = goal_sequence.loc[goal_sequence['event_type'].eq('Carry')]
shots_in_move = goal_sequence.loc[goal_sequence['event_type'].eq('Shot')]

# Completed passes and numbered pass markers.
for number, row in enumerate(passes.itertuples(index=False), start=1):
    plot_arrow(pitch_ax, row, COLORS['scotland'], linewidth=2.5 if number == 4 else 2.1)
    marker_x = row.start_x + (row.end_x - row.start_x) * 0.58
    marker_y = row.start_y + (row.end_y - row.start_y) * 0.58
    pitch_ax.scatter(marker_x, marker_y, s=195, color=COLORS['scotland'], edgecolor=COLORS['white'], linewidth=1.8, zorder=6)
    pitch_ax.text(marker_x, marker_y, str(number), color=COLORS['white'], fontsize=8.5, fontweight='bold', ha='center', va='center', zorder=7)

# Only show the carries that explain the tactical mechanism.
key_carries = carries.loc[carries['player'].isin(['Grant Campbell Hanley', 'Ben Doak'])]
for row in key_carries.itertuples(index=False):
    plot_arrow(pitch_ax, row, COLORS['carry'], linewidth=2.4, linestyle=(0, (2, 2)), zorder=4)

# Shots A and B.
adams_shot = shots_in_move.loc[shots_in_move['player'].eq('Che Adams')].iloc[0]
mcginn_shot = shots_in_move.loc[shots_in_move['player'].eq('John McGinn')].iloc[0]
plot_arrow(pitch_ax, adams_shot, COLORS['saved_shot'], linewidth=2.8, linestyle=(0, (2, 2)), zorder=5)
plot_arrow(pitch_ax, mcginn_shot, COLORS['goal'], linewidth=3.5, zorder=5)

for row, label, color in [(adams_shot, 'A', COLORS['saved_shot']), (mcginn_shot, 'B', COLORS['goal'])]:
    pitch_ax.scatter(row.start_x, row.start_y, s=270, color=color, edgecolor=COLORS['white'], linewidth=2, zorder=8)
    pitch_ax.text(row.start_x, row.start_y, label, color=COLORS['white'], fontsize=9, fontweight='bold', ha='center', va='center', zorder=9)

# High-contrast player labels with short leaders keep names clear of the paths.
labels = [
    (35.1, 35.2, 'McTominay', (-42, -14)),
    (43.3, 10.7, 'Robertson', (-18, 26)),
    (28.8, 33.1, 'Hendry', (-38, 18)),
    (48.3, 61.1, 'Hanley', (-18, -24)),
    (100.9, 69.1, 'Adams', (28, -14)),
    (113.6, 60.7, 'Doak', (0, -27)),
    (106.3, 42.2, 'McGinn', (-30, 24)),
]
for x, y, label, offset in labels:
    pitch_ax.annotate(
        label,
        xy=(x, y),
        xytext=offset,
        textcoords='offset points',
        fontsize=8.4,
        fontweight='bold',
        color=COLORS['ink'],
        ha='center',
        va='center',
        bbox=dict(boxstyle='round,pad=0.28', facecolor=COLORS['white'], edgecolor=COLORS['scotland'], linewidth=0.8, alpha=0.96),
        arrowprops=dict(arrowstyle='-', color=COLORS['muted'], linewidth=0.75, shrinkA=2, shrinkB=4),
        annotation_clip=False,
        zorder=10,
    )

# Header.
fig.text(0.04, 0.92, 'SIX PASSES, TWO SHOTS, ONE DECISIVE GOAL', fontsize=25, fontweight='bold', fontfamily=TITLE_FONT)
fig.text(0.04, 0.875, "How Scotland turned controlled circulation into McGinn's 28th-minute winner", fontsize=12, color=COLORS['muted'])

# Metric cards.
cards = [
    (0.04, '26', 'SECONDS', 'First receipt to goal'),
    (0.235, '6', 'PASSES', 'All six completed'),
    (0.43, '0.48', 'COMBINED xG', 'Adams shot + McGinn goal'),
]
for x, value, label, note in cards:
    card = FancyBboxPatch((x, 0.79), 0.17, 0.065, boxstyle='round,pad=0.008,rounding_size=0.015', transform=fig.transFigure, facecolor=COLORS['white'], edgecolor=COLORS['grid'], linewidth=1)
    fig.patches.append(card)
    fig.text(x + 0.014, 0.825, value, fontsize=17, fontweight='bold', color=COLORS['goal'])
    fig.text(x + 0.062, 0.832, label, fontsize=9, fontweight='bold')
    fig.text(x + 0.062, 0.805, note, fontsize=8.2, color=COLORS['muted'])

# Right-side explanatory panel.
panel = FancyBboxPatch((0, 0), 1, 1, boxstyle='round,pad=0.02,rounding_size=0.04', transform=side_ax.transAxes, facecolor=COLORS['white'], edgecolor=COLORS['grid'], linewidth=1, clip_on=False)
side_ax.add_patch(panel)
side_ax.text(0.08, 0.94, 'SEQUENCE ANATOMY', fontsize=17, fontweight='bold', fontfamily=TITLE_FONT, transform=side_ax.transAxes)

sequence_notes = [
    ('1', COLORS['scotland'], 'McTominay finds\nRobertson on the left'),
    ('2', COLORS['scotland'], 'Robertson recycles\nback to Hendry'),
    ('3', COLORS['scotland'], 'Hendry moves the ball\nacross to Hanley'),
    ('4', COLORS['scotland'], 'Hanley breaks the block:\n59-unit pass to Adams'),
    ('5', COLORS['scotland'], 'Adams links play\nand releases Doak'),
    ('6', COLORS['scotland'], 'Doak drives 23 units\nand returns the ball'),
    ('A', COLORS['saved_shot'], 'Adams: 0.40 xG\nsaved by Placide'),
    ('B', COLORS['goal'], 'McGinn attacks the rebound:\n0.07 xG, GOAL'),
]
y_positions = [0.84, 0.75, 0.66, 0.57, 0.48, 0.39, 0.285, 0.18]
for (label, color, note), y in zip(sequence_notes, y_positions):
    side_ax.scatter(0.12, y, s=125, color=color, transform=side_ax.transAxes, clip_on=False)
    side_ax.text(0.12, y, label, color=COLORS['white'], fontsize=7.5, fontweight='bold', ha='center', va='center', transform=side_ax.transAxes)
    side_ax.text(0.22, y, note, fontsize=8.7, fontweight='bold' if label == 'B' else 'normal', va='center', linespacing=1.25, transform=side_ax.transAxes)

side_ax.text(0.5, 0.07, "48% of Scotland's event-level xG", fontsize=9.5, fontweight='bold', color=COLORS['goal'], ha='center', transform=side_ax.transAxes, bbox=dict(boxstyle='round,pad=0.7', facecolor='#FFF2E8', edgecolor='none'))

# Legend and footer.
fig.text(0.04, 0.055, '━  Completed pass', color=COLORS['scotland'], fontsize=8.5)
fig.text(0.18, 0.055, '┅  Key carry', color=COLORS['carry'], fontsize=8.5)
fig.text(0.30, 0.055, '┅  Saved shot', color=COLORS['saved_shot'], fontsize=8.5)
fig.text(0.43, 0.055, '━  Goal', color=COLORS['goal'], fontsize=8.5)
fig.text(0.95, 0.055, 'El Mundial de los Datos', fontsize=8.5, color=COLORS['muted'], fontweight='bold', ha='right')

goal_output = export_jpg(fig, 'Haiti_vs_Scotland_Decisive_Goal_Sequence_Python.jpg')
plt.show()

## Validate the exported files

The university requires each image to be 1200 × 800 pixels and under 500 KB.

In [ ]:
validation = []
for image_path in [xg_output, goal_output]:
    with Image.open(image_path) as image:
        validation.append({
            'file': image_path.name,
            'width_px': image.width,
            'height_px': image.height,
            'size_kb': round(image_path.stat().st_size / 1024, 1),
            'dimensions_ok': image.size == (1200, 800),
            'under_500_kb': image_path.stat().st_size < 500 * 1024,
        })

validation_df = pd.DataFrame(validation)
display(validation_df)
assert validation_df['dimensions_ok'].all()
assert validation_df['under_500_kb'].all()
print('Both charts satisfy the publication requirements.')